# Intro 12 — Simple Linear Regression

Practice notebook for the **"Simple Linear Regression"** section of the stats course PDF.

## What you will learn

This notebook recaps the theory from the PDF section, then **verifies it with Python**.
Each part ends with a **"Your turn"** prompt; the full **Problem Set** at the end has
**click-to-reveal solutions**.

## How to use

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — statistics is about *relationships*, not memorized formulas.
3. LaTeX uses \( \) for inline math and \[ \] / $$ $$ for display math (KaTeX-friendly).

Let's begin.


---
## Setup — run this first

NumPy for numerics, SciPy for exact distributions, Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy import stats

np.random.seed(42)
rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", scipy.__version__)


---
## Part 1 — The model

Simple linear regression models a response \(Y\) in terms of a predictor \(X\):

$$
Y = \beta_0 + \beta_1 X + \varepsilon,
$$

where:

- \(\beta_0\) is the **intercept** (value of \(Y\) when \(X=0\)).
- \(\beta_1\) is the **slope** (change in \(Y\) per unit change in \(X\)).
- \(\varepsilon\) is a random error term with mean \(0\) and variance \(\sigma^2\).

Given data \((x_i, y_i)\), \(i=1,\dots,n\), the **least squares** estimators
\(\hat{\beta}_0, \hat{\beta}_1\) minimize the sum of squared residuals

$$
S(\beta_0,\beta_1) = \sum_{i=1}^{n} (y_i - \beta_0 - \beta_1 x_i)^2.
$$

We'll simulate data from a known line, then pretend we don't know the coefficients.


In [ ]:
# Ground-truth model: Y = b0 + b1*X + noise
b0_true, b1_true, sigma_true = 1.5, 0.8, 0.5
n = 60

rng = np.random.default_rng(42)
x = rng.uniform(0, 10, size=n)
y = b0_true + b1_true * x + rng.normal(0, sigma_true, size=n)

print(f"n = {n}, true intercept = {b0_true}, true slope = {b1_true}, noise sd = {sigma_true}")
print("First 5 (x, y):")
for xi, yi in list(zip(x, y))[:5]:
    print(f"  ({xi:6.3f}, {yi:6.3f})")


---
## Part 2 — Closed-form OLS estimates

Solving the normal equations yields

$$
\hat{\beta}_1 =
\frac{\sum_{i=1}^{n} (x_i - \bar{x})(y_i - \bar{y})}
     {\sum_{i=1}^{n} (x_i - \bar{x})^2},
\qquad
\hat{\beta}_0 = \bar{y} - \hat{\beta}_1 \bar{x}.
$$

The slope \(\hat{\beta}_1\) is the sample covariance of \(X\) and \(Y\) divided
by the variance of \(X\). Let's compute it from scratch with NumPy and verify it
against `np.polyfit`.


In [ ]:
# Closed-form OLS from the textbook formulas
x_bar = x.mean()
y_bar = y.mean()

Sxy = np.sum((x - x_bar) * (y - y_bar))   # numerator of slope
Sxx = np.sum((x - x_bar) ** 2)            # denominator of slope

b1_hat = Sxy / Sxx
b0_hat = y_bar - b1_hat * x_bar

print(f"From scratch:  b0_hat = {b0_hat:.6f},  b1_hat = {b1_hat:.6f}")
print(f"  Sxy = {Sxy:.6f},  Sxx = {Sxx:.6f}")

# Compare to np.polyfit (degree 1 -> [slope, intercept])
beta1_pf, beta0_pf = np.polyfit(x, y, deg=1)
print(f"np.polyfit:     b0_hat = {beta0_pf:.6f},  b1_hat = {beta1_pf:.6f}")

assert np.allclose([b0_hat, b1_hat], [beta0_pf, beta1_pf])
print("Match: closed-form OLS == np.polyfit ✔")


---
## Part 3 — Fitted values and the fitted line

The fitted model predicts

$$
\hat{y}_i = \hat{\beta}_0 + \hat{\beta}_1 x_i.
$$

We plot the data, the fitted line, and the residuals \(e_i = y_i - \hat{y}_i\).


In [ ]:
y_hat = b0_hat + b1_hat * x
residuals = y - y_hat

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: data + fitted line
axes[0].scatter(x, y, alpha=0.7, label="data $(x_i, y_i)$")
xs = np.linspace(x.min(), x.max(), 100)
axes[0].plot(xs, b0_hat + b1_hat * xs, color="crimson", lw=2,
             label=rf"fit: $\hat{{y}} = {b0_hat:.3f} + {b1_hat:.3f}\,x$")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")
axes[0].set_title("Simple linear regression fit")
axes[0].legend()

# Right: residuals
axes[1].scatter(y_hat, residuals, alpha=0.7)
axes[1].axhline(0, color="black", lw=1)
axes[1].set_xlabel(r"fitted $\hat{y}$")
axes[1].set_ylabel(r"residual $e_i = y_i - \hat{y}_i$")
axes[1].set_title("Residuals vs. fitted")

plt.tight_layout()
plt.show()

print(f"Sum of residuals (should be ~0): {residuals.sum():.2e}")
print(f"Mean residual: {residuals.mean():.2e}")


---
## Part 4 — Coefficient of determination \(R^2\)

The total sum of squares decomposes as

$$
\sum_{i=1}^{n} (y_i - \bar{y})^2
= \sum_{i=1}^{n} (\hat{y}_i - \bar{y})^2
+ \sum_{i=1}^{n} (y_i - \hat{y}_i)^2,
$$

i.e.

$$
\text{SST} = \text{SSR} + \text{SSE},
$$

where SST is the total sum of squares, SSR is the regression (explained) sum
of squares, and SSE is the residual (error) sum of squares. The coefficient of
determination is

$$
R^2 = \frac{\text{SSR}}{\text{SST}} = 1 - \frac{\text{SSE}}{\text{SST}}.
$$

It measures the fraction of variance in \(Y\) explained by the linear model.


In [ ]:
SST = np.sum((y - y_bar) ** 2)              # total
SSR = np.sum((y_hat - y_bar) ** 2)         # explained by regression
SSE = np.sum((y - y_hat) ** 2)             # residual / error

R2 = SSR / SST                              # = 1 - SSE / SST
R2_alt = 1 - SSE / SST

print(f"SST (total)            = {SST:.6f}")
print(f"SSR (regression)       = {SSR:.6f}")
print(f"SSE (residual)         = {SSE:.6f}")
print(f"Check SST = SSR + SSE: {np.isclose(SST, SSR + SSE)}  ({SSR + SSE:.6f})")
print(f"R^2  = SSR/SST         = {R2:.6f}")
print(f"R^2  = 1 - SSE/SST     = {R2_alt:.6f}")

# Cross-check via correlation: for simple linear regression R^2 = r^2
r = np.corrcoef(x, y)[0, 1]
print(f"r(x, y)                = {r:.6f}")
print(f"r^2                    = {r**2:.6f}")
assert np.isclose(R2, r**2)
print("Match: R^2 == r^2 ✔")


---
## Part 5 — Re-deriving the toy example

The PDF works a toy dataset by hand:

$$
(x, y) = (1,2),\ (2,3),\ (3,5),\ (4,4).
$$

With \(\bar{x} = 2.5\) and \(\bar{y} = 3.5\), the closed-form formulas give

$$
\hat{\beta}_1 = \frac{4.0}{5.0} = 0.8,
\qquad
\hat{\beta}_0 = 3.5 - 0.8 \cdot 2.5 = 1.5,
$$

so the fitted line is \(\hat{y} = 1.5 + 0.8\,x\). We reproduce it numerically
and check the \(R^2\) decomposition on this small dataset.


In [ ]:
x_toy = np.array([1.0, 2.0, 3.0, 4.0])
y_toy = np.array([2.0, 3.0, 5.0, 4.0])

xb, yb = x_toy.mean(), y_toy.mean()
b1_toy = np.sum((x_toy - xb) * (y_toy - yb)) / np.sum((x_toy - xb) ** 2)
b0_toy = yb - b1_toy * xb

print(f"xbar = {xb},  ybar = {yb}")
print(f"beta1_hat = {b1_toy}  (expected 0.8)")
print(f"beta0_hat = {b0_toy}  (expected 1.5)")

y_hat_toy = b0_toy + b1_toy * x_toy
SST_t = np.sum((y_toy - yb) ** 2)
SSR_t = np.sum((y_hat_toy - yb) ** 2)
SSE_t = np.sum((y_toy - y_hat_toy) ** 2)
R2_t = 1 - SSE_t / SST_t

print(f"SST = {SST_t},  SSR = {SSR_t:.6f},  SSE = {SSE_t:.6f}")
print(f"R^2 (toy) = {R2_t:.6f}")

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(x_toy, y_toy, zorder=3, label="toy data")
xs = np.linspace(x_toy.min() - 0.5, x_toy.max() + 0.5, 50)
ax.plot(xs, b0_toy + b1_toy * xs, color="crimson",
        label=rf"$\hat{{y}} = {b0_toy} + {b1_toy}\,x$")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title(f"Toy example: fitted line, $R^2 = {R2_t:.3f}$")
ax.legend()
plt.show()

assert np.isclose(b1_toy, 0.8) and np.isclose(b0_toy, 1.5)
print("Toy example matches the PDF ✔")


---
**Your turn:**

- Re-run Part 2 with a much smaller \(n\) (say 8). How do the estimates
  compare to the truth? Why?
- Add an outlier (a single \(y_i\) far off the line) and recompute
  \(\hat{\beta}_1\) and \(R^2\). Least squares is *not* robust — observe.
- Generate data where the true relationship is **nonlinear** (e.g.
  \(Y = 1 + 2X + 0.5 X^2 + \varepsilon\)) and fit a straight line. What happens
  to the residuals vs. fitted plot? What does that tell you about the model?


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. Given five points \((x, y) = (1,1),(2,3),(3,2),(4,5),(5,4)\), compute
\(\hat{\beta}_0\), \(\hat{\beta}_1\), and \(R^2\) using the closed-form formulas
(not `np.polyfit`).

2. Show that for simple linear regression the residuals sum to zero,
\(\sum_i e_i = 0\), and that the fitted values satisfy
\(\bar{\hat{y}} = \bar{y}\). Demonstrate it numerically on the simulated
dataset from Part 1.

3. Derive \(\hat{\beta}_1\) from the normal equation
\(\partial S/\partial \beta_1 = 0\) and verify your expression against
the covariance-over-variance form
\(\hat{\beta}_1 = \mathrm{Cov}(X,Y)/\mathrm{Var}(X)\) using `np.cov`.

4. Generate a dataset with true slope \(0\) (i.e. \(Y\) is pure noise
independent of \(X\)). Fit a line and report \(\hat{\beta}_1\) and \(R^2\).
What value of \(R^2\) do you expect? Why?

5. Write a function `ols(x, y)` returning \((\hat{\beta}_0, \hat{\beta}_1,
\text{SSE}, R^2)\). Test it on the toy dataset from the PDF and on a fresh
random sample of size 200 with \(\beta_0 = 3, \beta_1 = -2, \sigma = 1\).


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** Means: \(\bar{x}=3\), \(\bar{y}=3\). Numerator
\(\sum(x_i-\bar{x})(y_i-\bar{y}) = 10\). Denominator \(\sum(x_i-\bar{x})^2 = 10\).
So \(\hat{\beta}_1 = 1.0\), \(\hat{\beta}_0 = 3 - 1\cdot 3 = 0\).
Fitted \(\hat{y} = x\); \(R^2 = 1 - \text{SSE}/\text{SST} = 0.6\).

```python
x = np.array([1,2,3,4,5.0]); y = np.array([1,3,2,5,4.0])
xb, yb = x.mean(), y.mean()
b1 = np.sum((x-xb)*(y-yb)) / np.sum((x-xb)**2)
b0 = yb - b1*xb
yh = b0 + b1*x
R2 = 1 - np.sum((y-yh)**2) / np.sum((y-yb)**2)
print(b0, b1, R2)  # 0.0 1.0 0.6
```

**2.** By construction \(\hat{\beta}_0 = \bar{y} - \hat{\beta}_1\bar{x}\) implies
\(\bar{e} = \bar{y} - \bar{\hat{y}} = 0\), hence \(\sum e_i = 0\). Numerically:

```python
y_hat = b0_hat + b1_hat * x
e = y - y_hat
print(e.sum(), y_hat.mean(), y.mean())  # ~0, equal
```

**3.** Differentiating \(S = \sum(y_i - \beta_0 - \beta_1 x_i)^2\) w.r.t.
\(\beta_1\) and setting to zero gives
\(\sum x_i(y_i - \hat\beta_0 - \hat\beta_1 x_i) = 0\), which after substituting
\(\hat\beta_0 = \bar{y} - \hat\beta_1\bar{x}\) reduces to
\(\hat\beta_1 = \sum(x_i-\bar{x})(y_i-\bar{y})/\sum(x_i-\bar{x})^2
= \mathrm{Cov}(X,Y)/\mathrm{Var}(X)\).

```python
b1_cov = np.cov(x, y, ddof=0)[0,1] / np.var(x)
print(np.isclose(b1_cov, b1_hat))  # True
```

**4.** With true slope \(0\), \(\hat{\beta}_1\) should be near \(0\) and
\(R^2\) near \(0\) (the line explains almost none of the variance in \(Y\)).
On average over many replications \(E[R^2] \approx 1/(n-1)\) — small for large
\(n\), but occasionally a single fit can look spuriously explanatory.

```python
rng = np.random.default_rng(0)
x = rng.uniform(0,10,50); y = rng.normal(0,1,50)  # Y independent of X
b1 = np.sum((x-x.mean())*(y-y.mean()))/np.sum((x-x.mean())**2)
b0 = y.mean()-b1*x.mean()
R2 = 1-np.sum((y-(b0+b1*x))**2)/np.sum((y-y.mean())**2)
print(b1, R2)
```

**5.** A reusable OLS routine:

```python
def ols(x, y):
    xb, yb = x.mean(), y.mean()
    b1 = np.sum((x-xb)*(y-yb)) / np.sum((x-xb)**2)
    b0 = yb - b1*xb
    yh = b0 + b1*x
    sse = np.sum((y-yh)**2)
    R2 = 1 - sse / np.sum((y-yb)**2)
    return b0, b1, sse, R2

# Toy data from PDF
print(ols(np.array([1.,2,3,4]), np.array([2.,3,5,4])))
# Fresh sample
rng = np.random.default_rng(7)
x = rng.uniform(-5,5,200)
y = 3 - 2*x + rng.normal(0,1,200)
print(ols(x, y))  # b0 ~ 3, b1 ~ -2
```


</details>
